<a href="https://www.kaggle.com/code/tasneembinmahmood/human-emotion-detection-cv-92-accuracy?scriptVersionId=264678052" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

# Importing Library

In [ ]:
import tensorflow as tf
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from tensorflow.keras.layers import Input, Dense, GlobalAveragePooling2D
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.metrics import SparseCategoricalAccuracy
from sklearn.metrics import confusion_matrix, classification_report
from tensorflow.keras.layers import RandomFlip, RandomRotation, RandomZoom , RandomTranslation , RandomContrast , RandomBrightness
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.regularizers import l2

# Importing Dataset

In [ ]:
IMG_SIZE = (224, 224)
BATCH_SIZE = 16
SEED = 123
train_ds = tf.keras.utils.image_dataset_from_directory(
    "/kaggle/input/human-emotions-dataset-with-real-world-images/data/train",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='int',
    shuffle=True,
    validation_split=0.2,
    subset="training",
    seed=SEED
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    "/kaggle/input/human-emotions-dataset-with-real-world-images/data/train",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='int',
    shuffle=True,
    validation_split=0.2,
    subset="validation",
    seed=SEED
)

test_ds = tf.keras.utils.image_dataset_from_directory(
    "/kaggle/input/human-emotions-dataset-with-real-world-images/data/test",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='int',
    shuffle=False
)

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.prefetch(AUTOTUNE)
val_ds = val_ds.prefetch(AUTOTUNE)
test_ds = test_ds.prefetch(AUTOTUNE)

# Data Preprocessing

In [ ]:
data_augmentation = tf.keras.Sequential([
    RandomRotation(factor=(-0.025, 0.025)),
    RandomTranslation(height_factor=0.05, width_factor=0.05),
    RandomZoom(height_factor=(-0.05, 0.05), width_factor=(-0.05, 0.05)),
    RandomContrast(factor=0.1),
    RandomFlip(mode="horizontal"),
    RandomBrightness(factor=0.1)
], name="data_augmentation")

def cutmix_sparse(batch_x, batch_y, alpha=1.0):
    batch_size = tf.shape(batch_x)[0]

    indices = tf.random.shuffle(tf.range(batch_size))
    shuffled_x = tf.gather(batch_x, indices)
    shuffled_y = tf.gather(batch_y, indices)

    lam = tf.random.uniform([], 0, 1)

    H, W = IMG_SIZE
    r_x = tf.cast(W * tf.random.uniform([], 0, 1), tf.int32)
    r_y = tf.cast(H * tf.random.uniform([], 0, 1), tf.int32)
    r_w = tf.cast(W * tf.math.sqrt(1 - lam), tf.int32)
    r_h = tf.cast(H * tf.math.sqrt(1 - lam), tf.int32)

    x1 = tf.clip_by_value(r_x - r_w // 2, 0, W)
    y1 = tf.clip_by_value(r_y - r_h // 2, 0, H)
    x2 = tf.clip_by_value(r_x + r_w // 2, 0, W)
    y2 = tf.clip_by_value(r_y + r_h // 2, 0, H)

    mask = tf.ones([y2 - y1, x2 - x1, 3])
    pad_top = y1
    pad_left = x1
    pad_bottom = H - y2
    pad_right = W - x2
    mask = tf.pad(mask, [[pad_top, pad_bottom], [pad_left, pad_right], [0, 0]])
    mask = 1 - mask

    mixed_x = batch_x * mask + shuffled_x * (1 - mask)

    mixed_y = tf.where(lam > 0.5, batch_y, shuffled_y)

    return mixed_x, mixed_y


train_ds_augmented = train_ds.map(
    lambda x, y: cutmix_sparse(x, y), num_parallel_calls=AUTOTUNE
)


# Model Creation

In [ ]:
def build_model(input_shape=IMG_SIZE+(3,), num_classes=4):
    base_model = ResNet50(
        include_top=False,
        input_shape=input_shape,
        weights="imagenet"
    )
    base_model.trainable = False

    inputs = Input(shape=input_shape)
    x = data_augmentation(inputs)      
    x = base_model(x, training=False)    
    x = GlobalAveragePooling2D()(x)
    outputs = Dense(num_classes, activation="softmax", kernel_regularizer=l2(1e-4))(x)
    model = Model(inputs, outputs)

    lr_schedule = tf.keras.optimizers.schedules.CosineDecayRestarts(
        initial_learning_rate=1e-3,
        first_decay_steps=1000,
        t_mul=2.0,
        m_mul=0.8,
        alpha=1e-6
    )
    model.compile(
        optimizer=tf.keras.optimizers.AdamW(learning_rate=lr_schedule, weight_decay=1e-4),
        loss="sparse_categorical_crossentropy",
        metrics=[SparseCategoricalAccuracy(name="accuracy")]
    )
    return model, base_model

model, base_model = build_model()
model.summary()

# Training The Model

In [ ]:
history1 = model.fit(
    train_ds_augmented,
    validation_data=val_ds,
    epochs=10,
    callbacks=[
        EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True)
    ]
)

# Fine Tuning

## Total Layers of The Base Model

In [ ]:
print("Base model has", len(base_model.layers), "layers.")

## Tuning Process

In [ ]:
fine_tune_at = 100
for layer in base_model.layers[:fine_tune_at]:
    layer.trainable = False
for layer in base_model.layers[fine_tune_at:]:
    layer.trainable = True


model.compile(
    optimizer=tf.keras.optimizers.AdamW(learning_rate=1e-4, weight_decay=1e-5), 
    loss="sparse_categorical_crossentropy",
    metrics=[SparseCategoricalAccuracy(name="accuracy")]
)

history2 = model.fit(
    train_ds_augmented,
    validation_data=val_ds,
    epochs=25,
    callbacks=[
        ModelCheckpoint("best_model.keras", save_best_only=True, monitor="val_loss"),
        EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True),
        ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2, min_lr=1e-7)
    ]
)

# Evaluate

In [ ]:
loss, acc = model.evaluate(test_ds)
print("Test Accuracy:", acc)

# Confusion Matrix

In [ ]:
class_names = ['Angry', 'Happy', 'Sad', 'Surprise']
all_preds = []
all_labels = []

for images, labels in test_ds:
    preds = model.predict(images, verbose=0)  
    all_preds.append(preds)
    all_labels.append(labels.numpy())

all_preds = np.concatenate(all_preds, axis=0)
all_labels = np.concatenate(all_labels, axis=0)

y_true = all_labels
y_pred = np.argmax(all_preds, axis=1)

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=class_names, yticklabels=class_names)
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.title("Confusion Matrix")
plt.show()

print("Classification Report:\n")
print(classification_report(y_true, y_pred, target_names=class_names, digits=4))


# Plot

In [ ]:
def plot_actual_vs_predicted_batchwise(model, dataset, class_names, num_samples=16):
    for images, labels in dataset.take(1):
        preds = model.predict(images, verbose=0)
        images, labels, preds = images[:num_samples], labels[:num_samples], preds[:num_samples]

        rows = cols = int(np.ceil(np.sqrt(num_samples)))
        fig, axes = plt.subplots(rows, cols, figsize=(12, 12))
        axes = axes.flatten()

        pred_labels = np.argmax(preds, axis=1)

        for i in range(num_samples):
            img = images[i].numpy().astype("uint8")
            axes[i].imshow(img)
            true_label = class_names[labels[i]]
            predicted_label = class_names[pred_labels[i]]
            confidence = preds[i][pred_labels[i]]
            color = 'green' if labels[i] == pred_labels[i] else 'red'
            axes[i].set_title(
                f"Actual: {true_label}\nPred: {predicted_label}\nConf: {confidence*100:.1f}%",
                fontsize=9, color=color
            )
            axes[i].axis('off')

        for j in range(num_samples, len(axes)):
            axes[j].axis('off')

        fig.subplots_adjust(wspace=0.3, hspace=0.4)
        plt.show()

class_names = ['Angry', 'Happy', 'Sad', 'Surprise']
plot_actual_vs_predicted_batchwise(model, test_ds, class_names, num_samples=16)
